# AkidaNet on the real AKD1000 card vs the quantized Keras CNN, on ImageNet-C

Card version. The simulator version is `akidanet-cifar10.ipynb`. **No training.**

1. Load BrainChip's pretrained, already quantized AkidaNet (ImageNet, alpha 0.5, 224x224, Akida 1.0, 8-bit first layer, 4-bit weights and activations).
2. Convert it with `cnn2snn` to an Akida 1.0 model and save the `.fbz`.
3. Map it on the AKD1000, check the 10 BrainChip sample images, read power.
4. Evaluate on ImageNet-C (fog, Gaussian noise, contrast, severities 1 to 5) on the card. The quantized Keras model of the same weights is the CNN baseline. Writes `results_card/accuracy.csv`, `agreement.csv`, `energy.csv`, `predictions.csv`.

Akida numbers come from the real card only, never from the simulator. Run on the machine with the card, with nothing else holding `/dev/akida0`.

**Class scores: `predict`, not `forward`.** `forward` returns the raw int32 potentials of the last layer. `predict` returns them "obtained from the model discrete potentials by applying a shift and a scale" (akida API docs). The classifier has a per-class `act_step` (1000 values, 999 distinct, 24.7 to 108.3 on this model), so the argmax of `forward` is not the class. On the card, 100 fog-1 images: `forward` 0.23 top-1 / 0.38 agreement with Keras, `predict` 0.36 / 0.75 (Keras 0.40).

**Expected agreement.** Even with `predict`, card and Keras agree only ~0.75-0.8 on fog-1. The converted model run in software gives the same (0.787 on 2000 images, top-1 0.377 vs Keras 0.382). Disagreements are conversion rounding on images where Keras's top two classes are nearly tied (margin < 0.05: 0.54 agreement; > 0.2: 0.98+). Top-1 accuracy matches to within about half a point.

**Data:** ImageNet-C subset from `scripts/make_imagenet_subset.py`, layout `IMAGENET_C_DIR/<corruption>/<severity>/<wnid>/*.JPEG`. No clean ImageNet val images yet, so no clean baseline (BrainChip publishes 61.30% clean top-1, cited, not measured).

In [1]:
import os, re, csv, time
os.environ["TF_USE_LEGACY_KERAS"] = "1"   # MetaTF tools expect Keras 2

import numpy as np
import tensorflow as tf
import akida, cnn2snn, quantizeml, akida_models
from akida_models import akidanet_imagenet_pretrained
from akida_models.imagenet import get_preprocessed_samples
from cnn2snn import convert, set_akida_version, AkidaVersion

# cnn2snn targets Akida 2.0 by default. The AKD1000 is Akida 1.0, so load and convert inside
# `with set_akida_version(AkidaVersion.v1):` (as in the MetaTF AkidaNet/ImageNet tutorial).
for m in (tf, akida, cnn2snn, quantizeml, akida_models):
    print(m.__name__, getattr(m, "__version__", "?"))

2026-11-05 16:24:53.199976: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-11-05 16:24:53.202615: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-11-05 16:24:53.210840: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1793892293.225027  239563 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1793892293.229332  239563 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1793892293.240114  239563 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin

2026-11-05 16:24:55.326006: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


tensorflow 2.19.1
akida 2.19.3
cnn2snn 2.19.3
quantizeml 1.2.5
akida_models 1.14.2


In [2]:
# ---- config ----
ALPHA    = 0.5      # AkidaNet width; 0.5 at 224 is the one used in BrainChip's tutorial
IMG_SIZE = 224      # the pretrained quantized model is fixed at 224

IMAGENET_C_DIR   = "imagenet_subset/imagenet-c"   # on the server (~/akida-try); <root>/<corruption>/<severity>/<wnid>/*.JPEG
IMAGENET_VAL_DIR = None     # clean val images, same file names (<root>/<wnid>/*.JPEG); not available yet

CORRUPTIONS = ["fog", "gaussian_noise", "contrast"]
SEVERITIES  = [1, 2, 3, 4, 5]
N_PER_CLASS = 2     # images per class per condition -> 2000 images per condition
CHUNK       = 250   # images per card call; energy needs a batch of ~10+ (a single image is too brief for the sensor)
# Early stop after the first chunk (250 images, ~1 min): card vs Keras agreement below this means a bug.
# predict gives ~0.75-0.8 on fog-1; the wrong argmax(forward) gave 0.34-0.38.
MIN_AGREEMENT = 0.6
MAX_TOP1_GAP  = 0.03  # print a warning (no stop) if card and Keras top-1 differ more than this on a condition
RESULTS_DIR = "results_card"

## 1. Pretrained quantized model (BrainChip zoo)

In [3]:
with set_akida_version(AkidaVersion.v1):
    model_keras = akidanet_imagenet_pretrained(ALPHA, quantized=True, bitwidth=4)
print("input :", model_keras.input.dtype, model_keras.input_shape)
print("output:", model_keras.output_shape)

# BrainChip's 10 copyright-free sample images (224x224x3 uint8) as a first check of the Keras model
x_demo, y_demo = get_preprocessed_samples(IMG_SIZE, 3)
pots = model_keras.predict(x_demo, batch_size=100)
acc_keras_demo = np.mean(np.argmax(pots, 1) == y_demo)
print(f"TF-Keras accuracy on the {len(x_demo)} sample images: {acc_keras_demo:.2f}")

/home/moosebun2/miniconda3/envs/akida-env/lib/python3.11/site-packages/akida_models/model_io.py:147: UserWarning: Model akidanet_imagenet_224_alpha_50_iq8_wq4_aq4.h5 has been trained with akida_models 1.1.10 which is the last version supporting 1.0 models training. Continuing execution.
  warnings.warn(f'Model {model_name_v1} has been trained with akida_models 1.1.10 which is '


input : <dtype: 'float32'> (None, 224, 224, 3)
output: (None, 1000)


1/1 [==============================] - ETA: 0s

1/1 [==============================] - 1s 659ms/step


TF-Keras accuracy on the 10 sample images: 0.90


## 2. Convert to Akida 1.0

In [4]:
with set_akida_version(AkidaVersion.v1):
    model_akida = convert(model_keras)
print("akida IP version:", model_akida.ip_version)      # must be v1 for the AKD1000
model_akida.summary()
model_akida.save("akidanet_imagenet_a05_224_v1.fbz")
print("saved akidanet_imagenet_a05_224_v1.fbz")

akida IP version: IpVersion.v1
                 Model Summary                  
________________________________________________
Input shape    Output shape  Sequences  Layers
[224, 224, 3]  [1, 1, 1000]  1          15    
________________________________________________

_____________________________________________________________
Layer (type)              Output shape    Kernel shape     

============== SW/conv_0-classifier (Software) ==============

conv_0 (InputConv.)       [112, 112, 16]  (3, 3, 3, 16)    
_____________________________________________________________
conv_1 (Conv.)            [112, 112, 32]  (3, 3, 16, 32)   
_____________________________________________________________
conv_2 (Conv.)            [56, 56, 64]    (3, 3, 32, 64)   
_____________________________________________________________
conv_3 (Conv.)            [56, 56, 64]    (3, 3, 64, 64)   
_____________________________________________________________
separable_4 (Sep.Conv.)   [28, 28, 128]   (3, 3, 64, 

## 3. Map on the AKD1000 and check power (hardware check)

Power measurement works on the AKD1000 only and is off by default (BrainChip "See the power number" page). Without a card this notebook stops here on purpose: no simulator numbers.

In [5]:
devices = akida.devices()
print("Available devices:", [d.desc for d in devices])
assert devices, ("NO AKIDA DEVICE. This is the card version; use akidanet-cifar10.ipynb for the simulator. "
                 "If the card is installed, another process may hold it: fuser -v /dev/akida0")
device = devices[0]
assert device.version == akida.NSoC_v2, "This needs the AKD1000 (NSoC_v2)."
model_akida.map(device)
model_akida.summary()                       # NP allocation on the real chip

acc_akida_demo = model_akida.evaluate(x_demo, y_demo)
demo_pred = np.asarray(model_akida.predict(x_demo)).reshape(len(x_demo), -1).argmax(1)
print(f"AKD1000 on the {len(x_demo)} sample images: evaluate {acc_akida_demo:.2f}, "
      f"predict-argmax {np.mean(demo_pred == y_demo):.2f}, agreement with Keras {np.mean(demo_pred == np.argmax(pots, 1)):.2f} "
      "(BrainChip tutorial: 8/10)")

device.soc.power_measurement_enabled = True
_ = model_akida.predict(x_demo)
print(f"Floor power: {device.soc.power_meter.floor:.2f} mW")
print(model_akida.statistics)               # includes 'Last inference energy consumed (mJ/frame)'

Available devices: ['PCIe/NSoC_v2/0']


                                      Model Summary                                      
_________________________________________________________________________________________
Input shape    Output shape  Sequences  Layers  NPs  Skip DMAs  External Memory (Bytes)
[224, 224, 3]  [1, 1, 1000]  1          15      68   0          400000                 
_________________________________________________________________________________________

_________________________
Component (type)  Count
HRC               1    
_________________________
CNP1              67   
_________________________
FNP2              1    
_________________________

           External Memory Summary            
______________________________________________
Layer (type)         External Memory (Bytes)
classifier (Fully.)  400000                 
______________________________________________

_________________________________________________________________________
Layer (type)              Output shape    Kern

AKD1000 on the 10 sample images: evaluate 0.80, predict-argmax 0.80, agreement with Keras 0.90 (BrainChip tutorial: 8/10)
Floor power: 924.42 mW
Average framerate = 11.51 fps
Last inference power range (mW):  Avg 958.17 / Min 924.00 / Max 984.00 / Std 18.56 
Last inference energy consumed (mJ/frame): 83.26
Last inference clock: 146885656
Last program clock: 993897


## 4. ImageNet-C evaluation: quantized Keras (CNN baseline) vs Akida on the AKD1000

Every condition uses the same images: the first `N_PER_CLASS` files of each class. Class index = position of the class folder in sorted order (standard ImageNet order). Both models get the same uint8 batches of `CHUNK` images. The card is always called the same way (`predict` on chunks of 250, power meter on), so energy is comparable across conditions.

In [6]:
HAVE_C   = bool(IMAGENET_C_DIR)   and os.path.isdir(IMAGENET_C_DIR)
HAVE_VAL = bool(IMAGENET_VAL_DIR) and os.path.isdir(IMAGENET_VAL_DIR)

def build_sample(ref_dir, n_per_class):
    """(class_index, wnid, file name) for the first n files of every class folder in ref_dir."""
    wnids = sorted(d for d in os.listdir(ref_dir) if os.path.isdir(os.path.join(ref_dir, d)))
    assert len(wnids) == 1000, f"expected 1000 class folders in {ref_dir}, found {len(wnids)}: class indices would be wrong"
    sample = []
    for idx, wnid in enumerate(wnids):
        files = sorted(os.listdir(os.path.join(ref_dir, wnid)))[:n_per_class]
        sample += [(idx, wnid, f) for f in files]
    return sample

def load_image(path, clean):
    img = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    if clean:       # standard ImageNet eval: shorter side to 256, center crop 224
        shape = tf.cast(tf.shape(img)[:2], tf.float32)
        new_hw = tf.cast(tf.round(shape * (256.0 / tf.reduce_min(shape))), tf.int32)
        img = tf.image.resize(img, new_hw, antialias=True)
    img = tf.image.resize_with_crop_or_pad(img, IMG_SIZE, IMG_SIZE)   # ImageNet-C images are already 224x224
    return tf.cast(tf.round(tf.clip_by_value(tf.cast(img, tf.float32), 0.0, 255.0)), tf.uint8)

def make_ds(paths, labels, clean):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(lambda p, y: (load_image(p, clean), y), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(CHUNK).prefetch(tf.data.AUTOTUNE)

def read_energy():
    """mJ/frame of the last card inference, parsed from model.statistics ('Last inference energy consumed (mJ/frame)')."""
    m = re.search(r"energy consumed \(mJ/frame\):\s*([0-9.]+)", str(model_akida.statistics))
    return float(m.group(1)) if m else None

first_chunk_checked = False

def evaluate_condition(paths, labels, clean):
    """Same uint8 batch to both models. Card class = argmax of model_akida.predict (NOT forward, see the top)."""
    global first_chunk_checked
    keras_pred, akida_pred, energies = [], [], []
    for x, _ in make_ds(paths, labels, clean):
        x = x.numpy()
        keras_pred.append(np.asarray(model_keras.predict_on_batch(x)).reshape(len(x), -1).argmax(1))
        akida_pred.append(np.asarray(model_akida.predict(x)).reshape(len(x), -1).argmax(1))
        e = read_energy()
        if e is not None:
            energies.append(e)
        if not first_chunk_checked:
            agree = float(np.mean(akida_pred[0] == keras_pred[0]))
            print(f"first chunk ({len(x)} images): card vs Keras agreement {agree:.3f}, energy {e} mJ/frame")
            assert agree >= MIN_AGREEMENT, (f"card vs Keras agreement {agree:.3f} < {MIN_AGREEMENT} (expected ~0.75-0.8 on fog-1). "
                                            "Check that the class is argmax of model_akida.predict (not forward). Stopping.")
            first_chunk_checked = True
    return np.concatenate(keras_pred), np.concatenate(akida_pred), (float(np.mean(energies)) if energies else None)

In [7]:
rows_acc, rows_agree, rows_energy, rows_pred = [], [], [], []
ENERGY_SOURCE = f"AKD1000 power meter, model.statistics after model.predict on chunks of {CHUNK}, mJ/frame incl. floor power"

def write_csv(name, header, rows):
    os.makedirs(RESULTS_DIR, exist_ok=True)
    with open(os.path.join(RESULTS_DIR, name), "w", newline="") as f:
        w = csv.writer(f); w.writerow(header); w.writerows(rows)

def flush():
    write_csv("accuracy.csv", ["model", "corruption", "severity", "n_images", "top1"], rows_acc)
    write_csv("agreement.csv", ["corruption", "severity", "n_images", "agreement"], rows_agree)
    write_csv("energy.csv", ["model", "corruption", "severity", "energy_per_inference", "source"], rows_energy)
    write_csv("predictions.csv", ["image_id", "corruption", "severity", "label", "keras_pred", "akida_pred"], rows_pred)

if not HAVE_C:
    print("IMAGENET_C_DIR is not set (or not a folder): skipping step 4. Steps 1-3 are the hardware check.")
else:
    device.soc.power_measurement_enabled = True
    sample = build_sample(os.path.join(IMAGENET_C_DIR, CORRUPTIONS[0], str(SEVERITIES[0])), N_PER_CLASS)
    labels = np.array([s[0] for s in sample], dtype=np.int64)
    n = len(sample)
    print(f"{n} images per condition")

    conditions = ([("clean", 0)] if HAVE_VAL else []) + [(c, s) for c in CORRUPTIONS for s in SEVERITIES]
    for corruption, severity in conditions:
        clean = corruption == "clean"
        root = IMAGENET_VAL_DIR if clean else os.path.join(IMAGENET_C_DIR, corruption, str(severity))
        paths = [os.path.join(root, w, f) for _, w, f in sample]
        assert os.path.isfile(paths[0]), f"missing file: {paths[0]}"
        t0 = time.time()
        k_pred, a_pred, energy = evaluate_condition(paths, labels, clean)

        k_top1 = float(np.mean(k_pred == labels))
        a_top1 = float(np.mean(a_pred == labels))
        agree  = float(np.mean(a_pred == k_pred))
        rows_acc.append(["akidanet_keras_quantized", corruption, severity, n, f"{k_top1:.4f}"])
        rows_acc.append(["akidanet_akida_akd1000", corruption, severity, n, f"{a_top1:.4f}"])
        rows_agree.append([corruption, severity, n, f"{agree:.4f}"])
        rows_energy.append(["akidanet_akida_akd1000", corruption, severity,
                            "" if energy is None else f"{energy:.3f}",
                            ENERGY_SOURCE if energy is not None else "not reported by the SDK for this run"])
        for i, (_, w, f) in enumerate(sample):
            rows_pred.append([f"{w}/{f}", corruption, severity, int(labels[i]), int(k_pred[i]), int(a_pred[i])])
        print(f"{corruption:15s} sev {severity}: keras {k_top1:.3f} | akida card {a_top1:.3f} | agree {agree:.3f} "
              f"| energy {energy} mJ/frame  ({time.time() - t0:.0f}s)")
        if abs(a_top1 - k_top1) > MAX_TOP1_GAP:
            print(f"  WARNING: card vs Keras top-1 gap {a_top1 - k_top1:+.3f} is larger than {MAX_TOP1_GAP}")
        flush()                                      # keep this condition's rows even if the check below fails
        if clean or severity == 1:
            assert k_top1 > 0.2, ("Keras top-1 is far below the expected level. Check class folder order, file layout "
                                  "and preprocessing before trusting any number.")
    print("wrote", [os.path.join(RESULTS_DIR, f) for f in ("accuracy.csv", "agreement.csv", "energy.csv", "predictions.csv")])

2000 images per condition


first chunk (250 images): card vs Keras agreement 0.776, energy 83.81 mJ/frame


2026-11-05 16:28:21.496495: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


fog             sev 1: keras 0.382 | akida card 0.377 | agree 0.788 | energy 84.87 mJ/frame  (196s)


2026-11-05 16:31:37.613942: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


fog             sev 2: keras 0.277 | akida card 0.282 | agree 0.742 | energy 85.1175 mJ/frame  (196s)


... (11 outputs left out)

contrast        sev 2: keras 0.289 | akida card 0.293 | agree 0.751 | energy 84.495 mJ/frame  (195s)


contrast        sev 3: keras 0.130 | akida card 0.142 | agree 0.703 | energy 84.2575 mJ/frame  (193s)


contrast        sev 4: keras 0.029 | akida card 0.032 | agree 0.659 | energy 84.16874999999999 mJ/frame  (194s)


contrast        sev 5: keras 0.008 | akida card 0.007 | agree 0.629 | energy 83.925 mJ/frame  (194s)
wrote ['results_card/accuracy.csv', 'results_card/agreement.csv', 'results_card/energy.csv', 'results_card/predictions.csv']
